In [1]:
import time
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit

In [2]:
# Create SparkSession
spark = SparkSession.builder.master("local[*]").getOrCreate()
current_year = 2024

c:\Users\admin\miniconda3\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [3]:
# Read data again
start_time_read = time.time()
df_spark = spark.read.csv("./data/vehicles.csv", header=True, inferSchema=True)
end_time_read = time.time()
print(f"PySpark Reading Time: {end_time_read - start_time_read} seconds")


start_time = time.time()

# Drop unused columns
columns_to_drop = ['url', 'region', 'region_url', 'make', 'title_status', 'VIN', 'size', 'image_url', 'description', 'lat','long','county']
df_spark = df_spark.drop(*columns_to_drop)

# Calculate age based on the current year
df_spark = df_spark.withColumn("age", lit(current_year) - col("year"))

# Convert columns to appropriate data types
df_spark = df_spark.withColumn("price", col("price").cast("double"))
df_spark = df_spark.withColumn("year", col("year").cast("int"))
df_spark = df_spark.withColumn("odometer", col("odometer").cast("double"))

# Filter record with invalid year
df_spark = df_spark.filter((col("year") > 1900) & (col("year") <= 2024))

# Fill missing values with default values
df_spark = df_spark.fillna({
    "cylinders": "unknown",
    "fuel": "unknown",
    "transmission": "unknown",
    "drive": "unknown",
    "paint_color": "unknown",
    "type": "unknown"
})

end_time = time.time()
print(f"PySpark Processing Time: {end_time - start_time} seconds")

PySpark Reading Time: 16.561076164245605 seconds
PySpark Processing Time: 2.5744881629943848 seconds


In [4]:
import pandas as pd
import time

current_year = 2024

# Start timer for reading the data
start_time_read = time.time()

# Read the CSV file into a pandas DataFrame
df_pandas = pd.read_csv("./data/vehicles.csv")

end_time_read = time.time()
print(f"Pandas Reading Time: {end_time_read - start_time_read} seconds")

# Start timer for data processing
start_time_process = time.time()

# Drop unused columns
columns_to_drop = ['url', 'region', 'region_url', 'title_status', 'VIN', 'size', 'image_url', 'description', 'lat', 'long', 'county']
df_pandas = df_pandas.drop(columns=columns_to_drop)

# Calculate age based on the current year
df_pandas['age'] = current_year - df_pandas['year']

# Convert columns to appropriate data types
df_pandas['price'] = pd.to_numeric(df_pandas['price'], errors='coerce')
df_pandas['year'] = pd.to_numeric(df_pandas['year'], errors='coerce')
df_pandas['odometer'] = pd.to_numeric(df_pandas['odometer'], errors='coerce')

# Filter records with invalid year
df_pandas = df_pandas[(df_pandas['year'] > 1900) & (df_pandas['year'] <= 2024)]

# Fill missing values with default values
df_pandas['cylinders']=df_pandas['cylinders'].fillna('unknown')
df_pandas['fuel']=df_pandas['fuel'].fillna('unknown')
df_pandas['transmission']=df_pandas['transmission'].fillna('unknown')
df_pandas['drive']=df_pandas['drive'].fillna('unknown')
df_pandas['paint_color']=df_pandas['paint_color'].fillna('unknown')
df_pandas['type']=df_pandas['type'].fillna('unknown')

# End timer for data processing
end_time_process = time.time()

# Print processing time
print(f"Pandas Processing Time: {end_time_process - start_time_process} seconds")

Pandas Reading Time: 8.28257417678833 seconds
Pandas Processing Time: 0.2766258716583252 seconds
